<h1>Fine-Tuning SAM2 Image Predictor w/ CamVid</h1>

Credits:
- [Video](https://www.youtube.com/watch?v=bcwLbmALyLI)
- [Medium Article](https://medium.com/towards-data-science/train-fine-tune-segment-anything-2-sam-2-in-60-lines-of-code-928dd29a63b3)
- [Repository](https://github.com/sagieppel/fine-tune-train_segment_anything_2_in_60_lines_of_code/tree/main)

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt
from add_noise_to_images import noisify_image

In [ ]:
def load_data(dataset):
    data = []
    match dataset:
        case "CamVid":
            data_dir = "../CamVid/"

            for ff, name in enumerate(os.listdir(data_dir + "train/")):
                try:
                    data.append({
                        "image":data_dir + "train/"+name,
                        "annotation":data_dir+"train_labels/"+name[:-4]+"_L.png"
                    })
                except:
                    print("Encountered error with", name, ".")

        case "GTA5":
            data_dir = "../GTA5/"
            
            for ff, name in enumerate(os.listdir(data_dir + "images/")):
                try:
                    data.append({
                        "image":data_dir + "images/"+name,
                        "annotation":data_dir+"labels/"+name
                    })
                except:
                    print("Encountered error with", name, ".")

        case "bdd100k":
            data_dir = "../bdd100k/seg/"

            for ff, name in enumerate(os.listdir(data_dir + "images/train")):
                try:
                    data.append({
                        "image":data_dir + "images/train/"+name,
                        "annotation":data_dir+"color_labels/train/"+name[:-4]+"_train_color.png"
                    })
                except:
                    print("Encountered error with", name, ".")
        case _:
            FileNotFoundError("The dataset provided is not implemented.")

    return data


In [ ]:
def load_val_data(dataset):
    data = []
    match dataset:
        case "CamVid":
            data_dir = "../CamVid/"

            for ff, name in enumerate(os.listdir(data_dir + "val/")):
                try:
                    data.append({
                        "image":data_dir + "val/"+name,
                        "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
                    })
                except:
                    print("Encountered error with", name, ".")

        case "GTA5":
            data_dir = "../GTA5/"
            
            for ff, name in enumerate(os.listdir(data_dir + "images/")):
                try:
                    data.append({
                        "image":data_dir + "images/"+name,
                        "annotation":data_dir+"labels/"+name
                    })
                except:
                    print("Encountered error with", name, ".")

        case "bdd100k":
            data_dir = "../bdd100k/seg/"

            for ff, name in enumerate(os.listdir(data_dir + "images/val")):
                try:
                    data.append({
                        "image":data_dir + "images/val/"+name,
                        "annotation":data_dir+"color_labels/val/"+name[:-4]+"_train_color.png"
                    })
                except:
                    print("Encountered error with", name, ".")
        case _:
            FileNotFoundError("The dataset provided is not implemented.")

    return data

In [ ]:
data = load_data("bdd100k")

for pair in data[:5]:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def disp_np_array_as_img(img:np.array):
    plt.figure(figsize=(10, 10))
    plt.imshow(img)    
    plt.axis('off')
    plt.show()

In [ ]:
def read_batch(data): 
    # read random image and its annotatio from the CamVid dataset

    entry = data[np.random.randint(len(data))] # Choose a random entry
    img = cv2.imread(entry["image"])
    ann_map = cv2.imread(entry["annotation"])

    # Resize images to be 1024 px along one dim
    r = np.min([1024 / img.shape[1], 1024 / img.shape[0]]) # scalling factor
    img = cv2.resize(img, (int(img.shape[1] * r), int(img.shape[0] * r)))
    ann_map = cv2.resize(ann_map, (int(ann_map.shape[1] * r), int(ann_map.shape[0] * r)),interpolation=cv2.INTER_NEAREST)

    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    unique_colors = unique_colors = unique_colors[~np.all(unique_colors == [0,0,0], axis=1)]
    
    # Using the annotation, get all unique colors. Then, sort them into binary masks for each color.
    points = []
    masks = []

    for color in unique_colors:
        binary_mask = np.all(ann_map == color, axis=2).astype(np.uint8) # make binary mask
        # print(binary_mask.shape)
        # print(np.unique(binary_mask))

        # Get general shape - shape error
        mask = np.zeros(shape=(img.shape[0], img.shape[1], 3), dtype=np.uint8)

        mask[binary_mask == 1] = color
        # print(mask[binary_mask == 1].shape)
        masks.append(mask)
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))]) # choose random point/coordinate from mask
        points.append([[yx[1], yx[0]]]) # x,y
        
    return img, np.array(masks), np.array(points), np.ones([len(masks), 1])

if False: read_batch(data) # testing code

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
predictor = SAM2ImagePredictor(sam2_model) # load net

base_sam2 = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
base = SAM2ImagePredictor(base_sam2) # base sam2

In [ ]:
predictor.model.sam_mask_decoder.train(True) # enable training of mask decoder
predictor.model.sam_prompt_encoder.train(True) # enable training of prompt decoder

In [ ]:
optimizer = torch.optim.AdamW(
    params=predictor.model.parameters(), 
    lr=1e-5,
    weight_decay=4e-5
)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    # Resize images to 1024 x 1024
    r = np.min([1024 / image.shape[1], 1024 / image.shape[0]])
    image = cv2.resize(image, (int(image.shape[1] * r), int(image.shape[0] * r)))
    mask = cv2.resize(mask, (int(mask.shape[1] * r), int(mask.shape[0] * r)),interpolation=cv2.INTER_NEAREST)

    return image, mask

def get_points(mask, num_points): # Sample points inside the input mask
    points = []
    for i in range(num_points):
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))])
        points.append([[yx[1], yx[0]]])
    return np.array(points)

val = load_val_data("bdd100k")

for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))    

def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        # if len(scores) > 1:
        #     plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()

def show_masks_one_img(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True, random_color=True):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    for i, (mask, score) in enumerate(zip(masks, scores)):
        
        if random_color:
            color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
        else:
            color = np.array([30/255, 144/255, 255/255, 0.6])
        h, w = mask.shape[-2:]
        mask = mask.astype(np.uint8)
        mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
        if borders:
            import cv2
            contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
        plt.imshow(mask_image)
        
    plt.axis('off')
    plt.show()

    return mask_image

In [ ]:
%pip install ultralytics

In [ ]:
import pandas as pd

table = pd.read_csv("../download_dataset/class_dict.csv")
# print(table.head())

# dict based on english labels
class_dict = {row["name"].lower() : [row["r"] / 255, row["g"] / 255, row["b"] / 255] for _, row in table.iterrows()}
color_labels = [class_dict[key] for key in class_dict]

print("Class dict:")
print(class_dict)
label_map = {
    'person':'pedestrian'
}

def label_to_color(label:str):
    if label in class_dict:
            color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
    else:
        if label in label_map:
            label = label_map[label]
            if label in class_dict:
                color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
            else:
                print("Could not find", label)
                color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
        else:
            print("Could not find", label)
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    return color

In [ ]:
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')

In [ ]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))    

def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        # plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        if len(scores) > 1:
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()

In [ ]:
# predict masks
def test_predictor(predictor, use_noise=True):   
    pair = val[np.random.randint(low=0, high=len(val))] # update later to for-loop some images

    image_path, mask_path = pair["image"], pair["annotation"]
    image, mask = read_image(image_path, mask_path)
    gt = mask
    input_points = get_points(mask, num_points=30) # arbitrarily get 30 points
    input_labels = np.ones([input_points.shape[0],1])

    if use_noise: image = noisify_image(image, noise="random", threshold=0.1)

    with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0],1])
        )

        # show_masks(image, masks, scores, point_coords=input_points, input_labels=input_labels) # resolve mask shape issue

        masks=masks[:,0].astype(bool)
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
            predictor.set_image(image.copy()) # image encoder
            masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
                point_coords=input_points,
                point_labels=input_labels,
                multimask_output=True
            )

            masks=masks[:,0].astype(bool) # take one channel to convert RGB -> Bool
            shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

            show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

            # ------------- ANOTHER WAY TO SHOW MASKS -------------

            seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
            occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

            for i in range(shorted_masks.shape[0]):
                mask = shorted_masks[i]
                if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
                mask[occupancy_mask]=0
                seg_map[mask]=i+1
                occupancy_mask[mask]=1

            rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
            for id_class in range(1,seg_map.max()+1):
                rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

            print("Predicted")
            plt.imshow(rgb_image) # segmented
            plt.axis('off')
            plt.show()

            print("Mix")
            plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
            plt.axis('off')
            plt.show()

            print("Original")
            plt.imshow(image) # original image
            plt.axis('off')
            plt.show()

        print("Ground Truth")
        plt.imshow(gt) # original image
        plt.axis('off')
        plt.show()

    # return rgb_image


In [ ]:
def get_uniq_pixels(ann_map):
    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    return unique_colors

In [ ]:
def monte_carlo_uncertainty(predictor, image, mask, num_samples=10):
    # Use Monte Carlo dropout to estimate uncertainty

    input_points = get_points(mask, num_points=30) # arbitrarily get 30 points
    predictions = []

    for _ in range(num_samples):
        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0],1])
        )

        masks=masks[:,0].astype(bool)
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
        occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

        for i in range(shorted_masks.shape[0]):
            mask = shorted_masks[i]
            if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
            mask[occupancy_mask]=0
            seg_map[mask]=i+1
            occupancy_mask[mask]=1

        rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
        for id_class in range(1,seg_map.max()+1):
            rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

        predictions.append(rgb_image)
    
    predictions = np.stack(predictions, axis=0)
    mean_pred = np.mean(predictions, axis=0)
    variance = np.var(predictions, axis=0)

    return mean_pred, variance


In [ ]:
def color_to_id(image):
    """
    Replace each pixel in the input image with a unique ID representing its color.
    Pixels with the same color will have the same ID.

    Args:
        image (numpy.ndarray or Torch.Tensor): Input image as a 3D array/tensor (height, width, channels).

    Returns:
        numpy.ndarray or Torch.Tensor: Image where each pixel is replaced with a unique ID representing its color.
    """
    """
    Convert an RGB image to a mask ID image, where each unique color is assigned a unique ID.
    """
    if isinstance(image, torch.Tensor):
        if image.dim() != 3:
            raise ValueError("Input image must be a 3D tensor (height, width, channels).")
    
        # Reshape the image to a 2D tensor of pixels
        try:
            pixels = image.view(-1, 3)
        except:
            pixels = image.reshape(-1,3) # If tensors are non-contiguous

        # Create a dictionary to map unique colors to unique IDs
        unique_colors = {}
        current_id = 1  # Start IDs from 1 (0 can be reserved for background or unused)
        
        # Initialize a tensor to store the IDs
        id_tensor = torch.zeros(pixels.shape[0], dtype=torch.long, device=image.device)
        
        # Iterate over each pixel and assign a unique ID
        for i in range(pixels.shape[0]):
            color = tuple(pixels[i].tolist())  # Convert the pixel color to a tuple (hashable)
            if color not in unique_colors:
                unique_colors[color] = current_id
                current_id += 1
            id_tensor[i] = unique_colors[color]
        
        # Reshape back to the original image shape
        id_image = id_tensor.view(image.shape[:2])
        
        return id_image
    else:
        if image.ndim != 3:
            raise ValueError("Input image must be a 3D array (height, width, channels).")
    
        # Reshape the image to a 2D array of pixels
        pixels = image.reshape(-1, 3)
        
        # Find unique colors and assign IDs
        unique_colors, color_to_id = np.unique(pixels, axis=0, return_inverse=True)
        
        # Reshape back to the original image shape
        id_image = color_to_id.reshape(image.shape[:2])
    
        return id_image

# test code
image = torch.tensor([
    [[255, 0, 0], [255, 0, 0], [0, 255, 0], [0, 255, 0]],
    [[255, 0, 0], [255, 0, 0], [0, 255, 0], [0, 255, 0]],
    [[0, 0, 255], [0, 0, 255], [0, 0, 0], [0, 0, 0]],
    [[0, 0, 255], [0, 0, 255], [0, 0, 0], [0, 0, 0]]
], dtype=torch.float32, requires_grad=True)

# Convert colors to unique IDs
id_image = color_to_id(image)

# Note: to get a gradient, we need to run loss.backward() on the result.
# However, we have no model

print("Input Image (RGB):")
plt.imshow(image.cpu().detach().numpy())
plt.title("Torch colors")
plt.show() 

print("\nOutput Image (IDs):")
print(id_image)
plt.imshow((id_image / torch.max(id_image)).cpu().detach().numpy())
plt.title("Output")
plt.show() 

print("\nGradients of Input Image:")
print(image.grad)


image = np.random.randint(low=0, high=256, size=(400, 500, 3)) # note that values can only go up to high-1
print(image.shape)
id_example = color_to_id(image)
print(id_example.shape)
plt.imshow(image) 
plt.title("Numpy Random colors")
plt.show() 

plt.imshow(id_example)
plt.title("Classified based on ID")
plt.show()

In [ ]:
def old_calculate_total_iou(ground_truth, predicted):
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    """
    if isinstance(ground_truth, np.ndarray) and isinstance(predicted, np.ndarray):
        return np_calculate_total_iou(ground_truth, predicted)
    elif isinstance(ground_truth, torch.Tensor) and isinstance(predicted, torch.Tensor):
        return torch_calculate_total_iou(ground_truth, predicted)
    else:
        ValueError("Combination of input types not implemented yet")

def old_np_calculate_total_iou(ground_truth:np.ndarray, predicted:np.ndarray)->float:
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    """
    # Initialize a set to keep track of processed ground truth mask IDs
    processed_gt_ids = set()
    
    # Initialize the accumulator for total IoU
    total_iou = 0.0
    
    # Iterate over unique ground truth mask IDs
    for gt_id in np.unique(ground_truth):
        # Skip background or unlabeled pixels (assuming 0 is the background ID)
        if gt_id == 0:
            continue
        
        # Skip if this mask ID has already been processed
        if gt_id in processed_gt_ids:
            continue
        
        # Find all pixels in the ground truth with the current mask ID
        gt_mask = (ground_truth == gt_id)
        
        # Get the corresponding predicted mask ID for the first pixel in this ground truth mask
        # (assuming all pixels in the ground truth mask correspond to the same predicted mask ID)
        sample_pixel = np.where(gt_mask)
        if len(sample_pixel[0]) == 0:
            continue
        pred_id = predicted[sample_pixel[0][0], sample_pixel[1][0]]
        
        # Find all pixels in the predicted image with the corresponding mask ID
        pred_mask = (predicted == pred_id)
        
        # Mark this ground truth mask ID as processed
        processed_gt_ids.add(gt_id)
        
        # Calculate the intersection and union
        # intersection = np.logical_and(gt_mask, pred_mask).sum()
        # union = np.logical_or(gt_mask, pred_mask).sum()

        # Faster calculation of intersection and union
        intersection = (gt_mask * pred_mask).sum()  # Element-wise multiplication and sum
        union = (gt_mask + pred_mask).clip(0, 1).sum()  # Element-wise addition, clip to 1, and sum
        
        # Avoid division by zero
        if union == 0:
            continue
        
        # Calculate IoU for this mask pair and add to the accumulator
        total_iou += intersection / union
        # print(total_iou)
    
    return total_iou


def old_torch_calculate_total_iou(ground_truth: torch.Tensor, predicted: torch.Tensor):
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    The returned tensor will have the same gradient as the `predicted` tensor.

    Args:
        ground_truth (torch.Tensor): Ground truth mask ID image (2D tensor).
        predicted (torch.Tensor): Predicted mask ID image (2D tensor).

    Returns:
        torch.Tensor: Total IoU as a scalar tensor with gradients preserved.
    """
    # Ensure inputs are 2D tensors
    if ground_truth.dim() != 2 or predicted.dim() != 2:
        raise ValueError("Input tensors must be 2D (height, width).")
    
    # Initialize a set to keep track of processed ground truth mask IDs
    processed_gt_ids = set()
    
    # Initialize the accumulator for total IoU
    total_iou = torch.tensor(0.0, device=predicted.device, requires_grad=True)
    
    # Iterate over unique ground truth mask IDs
    for gt_id in torch.unique(ground_truth):
        # Skip background or unlabeled pixels (assuming 0 is the background ID)
        if gt_id == 0:
            continue
        
        # Skip if this mask ID has already been processed
        if gt_id in processed_gt_ids:
            continue
        
        # Find all pixels in the ground truth with the current mask ID
        gt_mask = (ground_truth == gt_id)
        
        # Get the corresponding predicted mask ID for the first pixel in this ground truth mask
        sample_pixel = torch.nonzero(gt_mask, as_tuple=True)
        if len(sample_pixel[0]) == 0:
            continue
        pred_id = predicted[sample_pixel[0][0], sample_pixel[1][0]]
        
        # Find all pixels in the predicted image with the corresponding mask ID
        pred_mask = (predicted == pred_id)
        
        # Mark this ground truth mask ID as processed
        processed_gt_ids.add(gt_id.item())
        
        # Calculate the intersection and union
        intersection = (gt_mask * pred_mask).sum().float()  # Element-wise multiplication and sum
        union = (gt_mask + pred_mask).clamp(0, 1).sum().float()  # Element-wise addition, clamp to 1, and sum
        
        # Avoid division by zero
        if union == 0:
            continue
        
        # Calculate IoU for this mask pair and add to the accumulator
        total_iou = total_iou + (intersection / union)
    
    return total_iou / len(torch.unique(ground_truth))

# Test Cases:
ground_truth = cv2.imread("../output_masks/example_image.png")[...,::-1]  # BGR -> RGB
ground_truth = color_to_id(ground_truth)
predicted = ground_truth     
total_iou = old_calculate_total_iou(ground_truth, predicted)
print("Total IoU 1:", total_iou) # should be an integer value

ground_truth = cv2.imread("../output_masks/example_image.png")[...,::-1]  # BGR -> RGB
ground_truth = color_to_id(ground_truth)
predicted = np.zeros(shape=ground_truth.shape) # pure black
total_iou = old_calculate_total_iou(ground_truth, predicted)
print("Total IoU 2:", total_iou) # should be 0 or close to 0

# Image size affects how fast IoU is calculated by A LOT.
ground_truth = np.random.randint(low=0, high=255, size=(300, 200, 3))
ground_truth = color_to_id(ground_truth)
predicted = np.random.randint(low=0, high=255, size=(300, 200, 3))
predicted = color_to_id(predicted)
total_iou = old_calculate_total_iou(ground_truth, predicted)
print("Total IoU 3:", total_iou) # should be some nonzero value

# Testing torch Tensor version
print("Torch Tensor Version:")
ground_truth = torch.tensor([
    [1, 1, 2, 2],
    [1, 1, 2, 2],
    [3, 3, 0, 0],
    [3, 3, 0, 0]
], dtype=torch.float)

predicted = torch.tensor([
    [1, 1, 2, 2],
    [1, 0, 2, 2],
    [3, 3, 0, 0],
    [3, 3, 0, 0]
], dtype=torch.float, requires_grad=True)

# Calculate total IoU
total_iou = old_calculate_total_iou(ground_truth, predicted)

# Perform a backward pass to compute gradients
total_iou.backward()

print("Total IoU:", total_iou.item())
print("Gradients of predicted:", predicted.grad)

In [ ]:
def calculate_total_iou(ground_truth, predicted):
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    """
    if isinstance(ground_truth, np.ndarray) and isinstance(predicted, np.ndarray):
        return np_calculate_total_iou(ground_truth, predicted)
    elif isinstance(ground_truth, torch.Tensor) and isinstance(predicted, torch.Tensor):
        return torch_calculate_total_iou(ground_truth, predicted)
    else:
        ValueError("Combination of input types not implemented yet")

def np_calculate_total_iou(ground_truth:np.ndarray, predicted:np.ndarray)->float:
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    """
    # Initialize a set to keep track of processed ground truth mask IDs
    processed_gt_ids = set()
    
    # Initialize the accumulator for total IoU
    total_iou = 0.0
    
    # Iterate over unique ground truth mask IDs
    for prd_id in np.unique(predicted):
        # Skip background or unlabeled pixels (assuming 0 is the background ID)
        if prd_id == 0:
            continue
        
        # Skip if this mask ID has already been processed
        if prd_id in processed_gt_ids:
            continue
        
        # Find all pixels in the ground truth with the current mask ID
        prd_mask = (predicted == prd_id)
        
        # Get the corresponding predicted mask ID for the first pixel in this ground truth mask
        # (assuming all pixels in the ground truth mask correspond to the same predicted mask ID)
        sample_pixel = np.where(prd_mask)
        if len(sample_pixel[0]) == 0:
            continue
        gt_id = ground_truth[sample_pixel[0][0], sample_pixel[1][0]]
        
        # Find all pixels in the predicted image with the corresponding mask ID
        gt_mask = (ground_truth == gt_id)
        
        # Mark this ground truth mask ID as processed
        processed_gt_ids.add(prd_id)
        
        # Calculate the intersection and union
        # intersection = np.logical_and(gt_mask, pred_mask).sum()
        # union = np.logical_or(gt_mask, pred_mask).sum()

        # Faster calculation of intersection and union
        intersection = (prd_mask * gt_mask).sum()  # Element-wise multiplication and sum
        union = (prd_mask * gt_mask).clip(0, 1).sum()  # Element-wise addition, clip to 1, and sum
        
        # Avoid division by zero
        if union == 0:
            continue
        
        # Calculate IoU for this mask pair and add to the accumulator
        total_iou += intersection / union
        # print(total_iou)
    
    return total_iou


def torch_calculate_total_iou(ground_truth: torch.Tensor, predicted: torch.Tensor):
    """
    Calculate the total IoU between a ground truth mask ID image and a predicted mask ID image.
    The returned tensor will have the same gradient as the `predicted` tensor.

    Args:
        ground_truth (torch.Tensor): Ground truth mask ID image (2D tensor).
        predicted (torch.Tensor): Predicted mask ID image (2D tensor).

    Returns:
        torch.Tensor: Total IoU as a scalar tensor with gradients preserved.
    """
    # Ensure inputs are 2D tensors
    if ground_truth.dim() != 2 or predicted.dim() != 2:
        raise ValueError("Input tensors must be 2D (height, width).")
    
    # Initialize a set to keep track of processed ground truth mask IDs
    processed_gt_ids = set()
    
    # Initialize the accumulator for total IoU
    total_iou = torch.tensor(0.0, device=predicted.device, requires_grad=True)
    
    # Iterate over unique ground truth mask IDs
    for pred_id in torch.unique(predicted):
        # Skip background or unlabeled pixels (assuming 0 is the background ID)
        if pred_id == 0:
            continue
        
        # Skip if this mask ID has already been processed
        if pred_id in processed_gt_ids:
            continue
        
        # Find all pixels in the predicted with the current mask ID
        pred_mask = (predicted == pred_id)
        
        # Get the corresponding ground truth ID for the first pixel in this predicted mask
        sample_pixel = torch.nonzero(pred_mask, as_tuple=True)
        if len(sample_pixel[0]) == 0:
            continue
        gt_id = ground_truth[sample_pixel[0][0], sample_pixel[1][0]]
        
        # Find all pixels in the ground truth image with the corresponding mask ID
        gt_mask = (ground_truth == gt_id)
        
        # Mark this predicted mask ID as processed
        processed_gt_ids.add(pred_id.item())
        
        # Calculate the intersection and union
        intersection = (pred_mask * gt_mask).sum().float()  # Element-wise multiplication and sum
        union = (pred_mask + gt_mask).clamp(0, 1).sum().float()  # Element-wise addition, clamp to 1, and sum
        
        # Avoid division by zero
        if union == 0:
            continue
        
        # Calculate IoU for this mask pair and add to the accumulator
        total_iou = total_iou + (intersection / union)
        
    
    return total_iou

# Test Cases:
ground_truth = cv2.imread("../output_masks/example_image.png")[...,::-1]  # BGR -> RGB
ground_truth = color_to_id(ground_truth)
predicted = ground_truth     
total_iou = calculate_total_iou(ground_truth, predicted)
print("Total IoU 1:", total_iou) # should be an integer value

ground_truth = cv2.imread("../output_masks/example_image.png")[...,::-1]  # BGR -> RGB
ground_truth = color_to_id(ground_truth)
predicted = np.zeros(shape=ground_truth.shape) # pure black
total_iou = calculate_total_iou(ground_truth, predicted)
print("Total IoU 2:", total_iou) # should be 0 or close to 0

# Image size affects how fast IoU is calculated by A LOT.
ground_truth = np.random.randint(low=0, high=255, size=(300, 200, 3))
ground_truth = color_to_id(ground_truth)
predicted = np.random.randint(low=0, high=255, size=(300, 200, 3))
predicted = color_to_id(predicted)
total_iou = calculate_total_iou(ground_truth, predicted)
print("Total IoU 3:", total_iou) # should be some nonzero value

# Testing torch Tensor version
print("Torch Tensor Version:")
ground_truth = torch.tensor([
    [1, 1, 2, 2],
    [1, 1, 2, 2],
    [3, 3, 0, 0],
    [3, 3, 0, 0]
], dtype=torch.float)

predicted = torch.tensor([
    [1, 1, 2, 2],
    [1, 0, 2, 2],
    [3, 3, 0, 0],
    [3, 3, 0, 0]
], dtype=torch.float, requires_grad=True)

# Calculate total IoU
total_iou = calculate_total_iou(ground_truth, predicted)

# Perform a backward pass to compute gradients
total_iou.backward()

print("Total IoU:", total_iou.item())
print("Gradients of predicted:", predicted.grad)

In [ ]:
def downsize_and_get_iou(predictor, image:np.ndarray, gt:np.ndarray):
    # Downsize the image, get the predictor's evaluation, and calcualte IoU
    # A necessity since using full IoU takes way too long

    # Resize the image to 256x256
    small_image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_LINEAR)

    # Resize the ground truth to 256x256
    small_gt = cv2.resize(gt, (256, 256), interpolation=cv2.INTER_LINEAR)

    input_points = get_points(gt, num_points=30) # arbitrarily get 30 points
    input_labels = np.ones([input_points.shape[0],1])

    with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0],1])
        )

        # show_masks(image, masks, scores, point_coords=input_points, input_labels=input_labels) # resolve mask shape issue

        masks=masks[:,0].astype(bool)
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        masks=masks[:,0].astype(bool) # take one channel to convert RGB -> Bool
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
        occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

        for i in range(shorted_masks.shape[0]):
            mask = shorted_masks[i]
            if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
            mask[occupancy_mask]=0
            seg_map[mask]=i+1
            occupancy_mask[mask]=1

        rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
        for id_class in range(1,seg_map.max()+1):
            rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

        gt_id = color_to_id(gt)
        prd_id = color_to_id(rgb_image)
        iou = calculate_total_iou(gt_id, prd_id)
        return iou

### Training Block

#### Training the Model

In [ ]:
# training loop
from datetime import datetime
import torch.nn.functional as F

sample=1
num_steps = 6000 # 6000, 500 before
# num_steps = 500 # below graph suggests improvements plateau at this range

all_loss = []
all_mse = []

for itr in range(1, int(num_steps) + 1):
    with torch.cuda.amp.autocast(): # cast to mix precision

        image, mask, input_point, input_label = read_batch(data)
        if mask.shape[0] == 0: continue # skip empty batches

        # ---------------------------------------- FINETUNED SAM2 PREDICTION ----------------------------------------
        predictor.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]

        low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
            image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),
            image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
            repeat_image=batched_mode,
            high_res_features=high_res_features,)
        
        prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

        # print(prd_masks.shape, mask.shape)
        prd_masks = prd_masks.permute(0,2,3,1)

        # ---------------------------------------- BASE SAM2 PREDICTION ----------------------------------------

        base.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = base._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in base._features["high_res_feats"]]

        low_res_masks, prd_scores, _, _ = base.model.sam_mask_decoder(
            image_embeddings=base._features["image_embed"][-1].unsqueeze(0),
            image_pe=base.model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
            repeat_image=batched_mode,
            high_res_features=high_res_features,)
        
        base_masks = base._transforms.postprocess_masks(low_res_masks, base._orig_hw[-1])# Upscale the masks to the original image resolution

        # print(prd_masks.shape, mask.shape)
        base_masks = base_masks.permute(0,2,3,1)

        # ---------------------------------------- LOSS CALCULATION ----------------------------------------

        # Initialize an empty tensor for the combined image
        prd_combined_image = torch.zeros_like(prd_masks[0], requires_grad=True)
        
        # Overlay each mask onto the combined image
        for prdd in prd_masks:
            prd_combined_image = torch.max(prd_combined_image, prdd)  # Use torch.max to overlay masks
        # # plt.imshow(combined_image.cpu().detach().numpy())
        # # plt.show()

        gt_combined_image = torch.zeros_like(prd_masks[0])
        for g in torch.tensor(mask).cuda(): # send tensor to cuda
            gt_combined_image = torch.max(gt_combined_image, g)  # Use torch.max to overlay masks

        # Convert both to 2D tensors & calculate IoU

        # Optional: Resize the images to 256x256 using bilinear interpolation
        gt_combined_image = gt_combined_image.permute(2, 1, 0) # For F.interpolate
        gt_combined_image = F.interpolate(gt_combined_image.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0) # batch size stuff
        gt_combined_image = gt_combined_image.permute(2, 1, 0) # C, H, W -> W, H, C
        
        prd_combined_image = prd_combined_image.permute(2, 1, 0) # For F.interpolate
        prd_combined_image = F.interpolate(prd_combined_image.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0) # batch size stuff
        prd_combined_image = prd_combined_image.permute(2, 1, 0) # C, H, W -> W, H, C
        
        # plt.imshow(gt_combined_image.cpu().detach().numpy())
        # plt.show()

        # for m in mask:
        #     plt.imshow(m)
        #     plt.show()


        gt_id = color_to_id(gt_combined_image)
        prd_id = color_to_id(prd_combined_image)
        iou = calculate_total_iou(gt_id, prd_id)
        iou_loss = 1 - iou

        # iou_loss = 1 - downsize_and_get_iou(predictor, image=image, gt=gt_combined_image)

        # print("Total IoU:", total_iou.item())
        # print("Gradients of predicted:", predicted.grad)
        
        # segmentation loss calculation
        gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
        
        prd_mask = torch.sigmoid(prd_masks) # Turn logit map to probability map

        seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss

        # mse = torch.mean((prd_masks - base_masks) ** 2)

        _, var = monte_carlo_uncertainty(predictor=predictor, image=image, mask=mask, num_samples=10)
        uncertainty_loss = (torch.Tensor(var).to(device="cuda") / 255.0) * (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001))
        uncertainty_loss_mean, uncertainty_loss_std = torch.mean(uncertainty_loss), torch.std(uncertainty_loss)
        uncertainty_loss = (uncertainty_loss - uncertainty_loss_mean) / uncertainty_loss_std
        
        # Monte-Carlo Uncertainty based loss
        # loss = uncertainty_loss
        
        # Naive Approach Loss
        # loss = seg_loss + mse 

        # Naive + IoU - High IoU is typially good
        # loss = seg_loss + mse + iou_loss

        # Mix
        # loss = seg_loss + mse + uncertainty_loss + iou_loss
        loss = 0.7*seg_loss

        # Model is not learning to segment whole objects (cars, people, etc.) - WE NEED IOU

        # backpropogate loss
        predictor.model.zero_grad() # empty gradient        
        scaler.scale(loss).backward()  # Backpropogate - only requirement is that it is a tensor with a gradient
        
        scaler.step(optimizer)
        scaler.update() # Mix precision

        if itr%10==0: # Used to be 10
            test_predictor(predictor, use_noise=False)
            print(f"Confidence Scores={prd_scores.cpu().detach().numpy()}")

    
        # Display results
        # if itr==1: mean_iou=0
        # detached_iou=np.mean(iou.cpu().detach().numpy())
        # mean_iou = mean_iou * 0.99 + 0.01 * detached_iou

        scalar_loss = loss.cpu().detach()
        # print(f"step {itr}) | Loss={scalar_loss} | MSE between base and finetuned={mse} | Monto-Carlo Uncertainty Weighted Loss={uncertainty_loss}")
        # print(f"step {itr}) | Loss={scalar_loss} | MSE between base and finetuned={mse}")
        print(f"step {itr}) | Loss={scalar_loss} | Seg Loss={seg_loss.item()} | IoU Loss={iou_loss.item()} | Monto-Carlo Uncertainty Weighted Loss={uncertainty_loss}")
        all_loss.append(scalar_loss)
        # all_mse.append(mse.cpu().detach()) # NEVER STORE LIST OF TENSORS. DESTROYS RAM.

        if itr%500==0:
            # Save model
            date_str = str(datetime.now()).replace(":","-")
            torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
            print("saved model.")

date_str = str(datetime.now()).replace(":","-")
torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
print("saved model.")


#### Graphing Loss Values Over Time

In [ ]:
x_values = list(range(len(all_loss)))
all_loss = [e.cpu().detach() for e in all_loss]
all_mse = [e.cpu().detach() for e in all_mse]

plt.figure(figsize=(20, 10))
plt.plot(x_values, all_loss, label='Overall Loss', marker='o')  
plt.plot(x_values, all_mse, label='MSE between finetuned SAM2 and base SAM2', marker='s')  

# Add labels, title, and legend
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss and MSE Over Training Steps')
plt.legend()

# Display the graph
plt.show()

6000 steps may be overkill. 500 steps looks like enough.

### Testing on some Images

In [ ]:
def load_val_data(dataset):
    data = []
    match dataset:
        case "CamVid":
            data_dir = "../CamVid/"

            for ff, name in enumerate(os.listdir(data_dir + "val/")):
                try:
                    data.append({
                        "image":data_dir + "val/"+name,
                        "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
                    })
                except:
                    print("Encountered error with", name, ".")

        case "GTA5":
            data_dir = "../GTA5/"
            
            for ff, name in enumerate(os.listdir(data_dir + "images/")):
                try:
                    data.append({
                        "image":data_dir + "images/"+name,
                        "annotation":data_dir+"labels/"+name
                    })
                except:
                    print("Encountered error with", name, ".")

        case "bdd100k":
            data_dir = "../bdd100k/seg/"

            for ff, name in enumerate(os.listdir(data_dir + "images/val")):
                try:
                    data.append({
                        "image":data_dir + "images/val/"+name,
                        "annotation":data_dir+"color_labels/val/"+name[:-4]+"_train_color.png"
                    })
                except:
                    print("Encountered error with", name, ".")
        case _:
            FileNotFoundError("The dataset provided is not implemented.")

    return data

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt

# Validate our finetuned SAM2

# use bfloat16 for memory efficiency
torch.autocast(device_type="cuda", dtype=torch.float32).__enter__()

# load some example images
# data_dir = "../CamVid/"
# val = []
# for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
#     try:
#         val.append({
#             "image":data_dir + "val/"+name,
#             "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
#         })
#     except:
#         print("Encountered error with", name, ".")


# for pair in val:
#     print(pair["image"], pair["annotation"]) # to test above works


# data_dir = "../GTA5/"
# val = []
# for ff, name in enumerate(os.listdir(data_dir + "images/")):
#     try:
#         val.append({
#             "image":data_dir + "images/"+name,
#             "annotation":data_dir+"labels/"+name
#         })
#     except:
#         print("Encountered error with", name, ".")

val = load_val_data("bdd100k")
for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    r = np.min([1024 / image.shape[1], 1024 / image.shape[0]]) # scalling factor
    image = cv2.resize(image, (int(image.shape[1] * r), int(image.shape[0] * r)))
    mask = cv2.resize(mask, (int(mask.shape[1] * r), int(mask.shape[0] * r)),interpolation=cv2.INTER_NEAREST)


    return image, mask

def get_points(mask, num_points, random=True): # Sample points inside the input mask
    points = []
    coords = np.argwhere(mask > 0)
    if random:
        for i in range(num_points):
            yx = np.array(coords[np.random.randint(len(coords))])
            points.append([[yx[1], yx[0]]])
    else:
        n = len(coords)
        stride = max(n // num_points, 1)
        for i in range(0, n, stride):
            yx = coords[i]
            points.append([[yx[1], yx[0]]])
    return np.array(points)

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

#### Finetuned SAM2

In [ ]:
import numpy as np
import torch
import cv2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

model_dir = "../models/"

sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

# build finetuned model and load weights
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))
print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

base_sam2 = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
base = SAM2ImagePredictor(base_sam2)

# use bfloat16 for the entire script (memory efficient)
torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
from add_noise_to_images import noisify_image

pair = val[np.random.randint(len(val))] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]

def read_image(image_path, mask_path): # read and resize image and mask
    img = cv2.imread(image_path)[...,::-1]  # read image as rgb
    mask = cv2.imread(mask_path,0) # mask of the region we want to segment

    return img, mask

image, mask = read_image(image_path, mask_path)
# The test images look kinda dark - maybe increase brightness?

# brightness_factor = 90
# image += brightness_factor

# if image.dtype == np.uint8:
#     image = np.clip(image, 0, 255)
# else:
#     image = np.clip(image, 0.0, 1.0)

gt = mask
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points
input_labels = np.ones([input_points.shape[0],1])
# image = noisify_image(image, noise="random", threshold=0.1)

sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt" # trying large models since we are dealing with larger data & larger images 
                                                        # all models before 2/26/2025 3:47 PM EST should be small ones
                                                        # Note that the small ones seem to do exceptionally well on images of size 720 x 960
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))

print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=True
    )

    masks=masks[:,0].astype(bool) # take one channel to convert RGB -> Bool
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

#### Base SAM2

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    base.set_image(image.copy()) # image encoder
    masks, scores, logits = base.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

In [ ]:

with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)    
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

#### 6000 step BDD100K IoU Finetuned SAM2

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=True
    )

    masks=masks[:,0].astype(bool) # take one channel to convert RGB -> Bool
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        # Iterate through individual masks, giving each a color for visual representation
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

#### BDD 100K IoU Finetuned SAM2

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt" # trying large models since we are dealing with larger data & larger images 
                                                        # all models before 2/26/2025 3:47 PM EST should be small ones
                                                        # Note that the small ones seem to do exceptionally well on images of size 720 x 960
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load("../final_models/large_models/(6000 Old IoU + Naive trained) model-2025-03-02 21-24-22.314668"))

print("Loaded model:", "../final_models/model-2025-02-18 10-20-11.080153.torch")

with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)
    
    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

#### GTA-Trained 6000 Step Finetuned Naive Approach Large SAM2

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

# build finetuned model and load weights
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load("../final_models/large_models/model-2025-02-26 19-53-10.408176.torch"))
print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

base_sam2 = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
base = SAM2ImagePredictor(base_sam2)

# use bfloat16 for the entire script (memory efficient)
torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=True
    )

    masks=masks[:,0].astype(bool) # take one channel to convert RGB -> Bool
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)    
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

#### CamVid-Trained 6000 Step Finetuned Naive Approach Small SAM2:

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt" # trying large models since we are dealing with larger data & larger images 
                                                        # all models before 2/26/2025 3:47 PM EST should be small ones
                                                        # Note that the small ones seem to do exceptionally well on images of size 720 x 960
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load("../final_models/small_models/model-2025-02-18 10-20-11.080153.torch"))

print("Loaded model:", "../final_models/model-2025-02-18 10-20-11.080153.torch")

with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)
    
    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)


In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)

    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        # Iterate through individual masks, giving each a color for visual representation
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

#### CamVid-Trained 500 Step Finetuned Naive Approach Small SAM2

In [ ]:
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load("../final_models/small_models/model-2025-02-23 22-58-27.138582.torch"))

print("Loaded model:", "../final_models/model-2025-02-23 22-58-27.138582.torch")

with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)
    show_masks_one_img(image=image, masks=masks, scores=scores, point_coords=input_points, input_labels=input_labels)


In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        # Iterate through individual masks, giving each a color for visual representation
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    print("Predicted")
    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    print("Mix")
    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    print("Original")
    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()

    print("Ground Truth")
    plt.imshow(gt) # original image
    plt.axis('off')
    plt.show()

# SCRAP WORK BELOW - USE IF NEEDED:

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

predictor.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get 30 points
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = predictor.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=True  # Set to True if you want multiple masks
)
# print("output shape", masks[0].shape)

plt.figure(figsize=(10,10))
plt.axis('off')
plt.imshow(masks[0])

From the SAM2 Automatic Mask Generator Notebook:

```Since SAM 2 can efficiently process prompts, masks for the entire image can be generated by sampling a large number of prompts over an image.```

```The class SAM2AutomaticMaskGenerator implements this capability. It works by sampling single-point input prompts in a grid over the image, from each of which SAM can predict multiple masks. Then, masks are filtered for quality and deduplicated using non-maximal suppression. Additional options allow for further improvement of mask quality and quantity, such as running prediction on multiple crops of the image or postprocessing masks to remove small disconnected regions and holes.```

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

predictor.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get a bunch of points - 4000 seems to be a good number
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = predictor.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=False  # Set to True if you want multiple masks
)
# print("output shape", masks[0].shape)

for mask in masks:
    plt.figure(figsize=(10,10))
    plt.axis('off')
    plt.imshow(mask)

Base SAM2


In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

base.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get 30 points
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = base.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=True  # Set to True if you want multiple masks
)
print("output shape", masks[0].shape)

plt.figure(figsize=(10,10))
plt.axis('off')
plt.imshow(masks[0])